In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/raw/credit_portfolio.csv"
)

df["observation_date"] = pd.to_datetime(
    df["observation_date"]
)

In [2]:
df["ead"].describe()

count    240000.000000
mean      11668.219125
std        7878.669406
min         473.459456
25%        6369.942240
50%        9645.868592
75%       14650.325319
max       95407.932542
Name: ead, dtype: float64

In [3]:
df[
    [
        "loan_amount",
        "ead"
    ]
].head()

,loan_amount,ead
0,20319.398492,14575.750761
1,9298.951400,6616.637750
2,13670.088177,9430.217798
3,8641.969081,5509.788429
4,13499.049134,8704.152055


In [4]:
ead_features = [
    "age",
    "annual_income",
    "credit_score",
    "debt_to_income",
    "credit_utilization",
    "loan_amount",
    "loan_term_months",
    "interest_rate",
    "previous_defaults",
    "delinquencies_12m"
]

In [5]:
ead_target = "ead"

In [6]:
ead_train = df[
    df["observation_date"]
    < "2023-07-01"
].copy()

ead_val = df[
    (
        df["observation_date"]
        >= "2023-07-01"
    )
    &
    (
        df["observation_date"]
        < "2023-10-01"
    )
].copy()

ead_oot = df[
    df["observation_date"]
    >= "2023-10-01"
].copy()

In [7]:
X_ead_train = ead_train[
    ead_features
]

y_ead_train = ead_train[
    "ead"
]

X_ead_oot = ead_oot[
    ead_features
]

y_ead_oot = ead_oot[
    "ead"
]

In [11]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    RandomForestRegressor
)

ead_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

ead_model = Pipeline(
    steps=[
        (
            "preprocessor",
            ead_preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                max_depth=6,
                min_samples_leaf=30,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [12]:
ead_model.fit(
    X_ead_train,
    y_ead_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['age','annual_income','credit_score',...,'interest_rate', 'previous_defaults','delinquencies_12m']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantage

In [21]:
import joblib
from pathlib import Path

models_dir = Path("../models")

models_dir.mkdir(
    exist_ok=True
)

joblib.dump(
    ead_model,
    models_dir / "ead_model.joblib"
)

['..\\models\\ead_model.joblib']

In [15]:
ead_predictions = (
    ead_model.predict(
        X_ead_oot
    )
)

In [14]:
ead_predictions = np.maximum(
    ead_predictions,
    0
)

In [18]:
from sklearn.metrics import mean_absolute_error,mean_squared_error
ead_mae = mean_absolute_error(
    y_ead_oot,
    ead_predictions
)

ead_rmse = np.sqrt(
    mean_squared_error(
        y_ead_oot,
        ead_predictions
    )
)

print(
    f"EAD MAE: {ead_mae:,.2f}"
)

print(
    f"EAD RMSE: {ead_rmse:,.2f}"
)

EAD MAE: 1,435.89
EAD RMSE: 2,142.43


In [19]:
ead_comparison = pd.DataFrame({
    "actual_ead": y_ead_oot.values,
    "predicted_ead": ead_predictions
})

ead_comparison.head()

,actual_ead,predicted_ead
0,6801.007466,8373.181314
1,20079.073155,17731.262397
2,5413.617951,3952.351414
3,41174.242281,37763.597981
4,3763.399339,3407.125559


In [20]:
ead_comparison.to_csv(
    "../data/outputs/ead_predictions.csv",
    index=False
)